In [3]:
%pip install gensim

  Using cached gensim-4.4.0.tar.gz (23.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-2.4.0-cp314-cp314-win_amd64.whl.metadata (7.6 kB)
Using cached smart_open-8.0.1-py3-none-any.whl (73 kB)
Using cached wrapt-2.4.0-cp314-cp314-win_amd64.whl (96 kB)
Failed to build gensim
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [806 lines of output]
      C:\Users\Abhi\AppData\Local\Temp\pip-build-env-c3_vchg7\overlay\Lib\site-packages\setuptools\_distutils\dist.py:318: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\Abhi\AppData\Local\Temp\pip-build-env-c3_vchg7\overlay\Lib\site-packages\setuptools\_distutils\dist.py:318: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\gensim
      copying gensim\downloader.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\interfaces.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\matutils.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\nosy.py -> build\lib.win-amd64-cpython-314\gensim
     

In [1]:
import sys
print(sys.executable)

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv312\Scripts\python.exe


In [2]:
import gensim
print(gensim.__version__)

from gensim.models import Word2Vec
print("Word2Vec import successful")

4.4.0
Word2Vec import successful


In [3]:
import pandas as pd
import numpy as np
import optuna

from gensim.models import Word2Vec
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

In [4]:
dataset = pd.read_csv("../data/processed/processed_comments.csv")

print(dataset.shape)
dataset.head()

(36793, 2)


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [5]:
cleaned_dataset = dataset.dropna(
    subset=["clean_comment", "category"]
).copy()

X_cleaned = cleaned_dataset["clean_comment"]
y_cleaned = cleaned_dataset["category"]

print("Dataset shape:", cleaned_dataset.shape)
print("\nClass distribution:")
print(y_cleaned.value_counts())

Dataset shape: (36662, 2)

Class distribution:
category
 1    15770
 0    12644
-1     8248
Name: count, dtype: int64


In [6]:
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(
    X_cleaned,
    y_cleaned,
    test_size=0.2,
    random_state=42,
    stratify=y_cleaned
)

print("Train:", X_train_cleaned.shape)
print("Test:", X_test_cleaned.shape)

Train: (29329,)
Test: (7333,)


In [7]:
X_train_tokenized = [
    sentence.split()
    for sentence in X_train_cleaned
]

X_test_tokenized = [
    sentence.split()
    for sentence in X_test_cleaned
]

print(X_train_tokenized[0][:20])

['bjp', 'done', 'something', 'one', 'advertising', 'marketing', 'world', 'could', 'even', 'think', 'marketing', 'selling', 'stuff', 'inflating', 'value', 'make', 'potential', 'customer', 'believe', 'usefulness']


In [8]:
word2vec_model = Word2Vec(
    sentences=X_train_tokenized,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

print("Vocabulary size:", len(word2vec_model.wv))
print("Vector size:", word2vec_model.vector_size)

Vocabulary size: 42477
Vector size: 100


In [9]:
def vectorize_comments(tokenized_comments, word2vec_model):

    vectorized_comments = []

    for tokens in tokenized_comments:

        vectors = [
            word2vec_model.wv[token]
            for token in tokens
            if token in word2vec_model.wv
        ]

        if len(vectors) > 0:
            comment_vector = np.mean(vectors, axis=0)
        else:
            comment_vector = np.zeros(
                word2vec_model.vector_size
            )

        vectorized_comments.append(comment_vector)

    return np.array(vectorized_comments)

In [10]:
X_train_word2vec = vectorize_comments(
    X_train_tokenized,
    word2vec_model
)

X_test_word2vec = vectorize_comments(
    X_test_tokenized,
    word2vec_model
)

print("Train:", X_train_word2vec.shape)
print("Test:", X_test_word2vec.shape)

Train: (29329, 100)
Test: (7333, 100)


In [11]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(
    y_train_cleaned
)

y_test_encoded = label_encoder.transform(
    y_test_cleaned
)

print("Original labels:", label_encoder.classes_)
print("Encoded labels:", np.unique(y_train_encoded))

Original labels: [-1  0  1]
Encoded labels: [0 1 2]


In [12]:
def objective(trial):

    params = {
        "objective": "multiclass",
        "num_class": 3,

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            20
        ),

        "reg_alpha": 0.1,
        "reg_lambda": 0.1,

        "class_weight": "balanced",
        "random_state": 42,
        "verbosity": -1
    }

    model = LGBMClassifier(**params)

    scores = cross_val_score(
        model,
        X_train_word2vec,
        y_train_encoded,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    return scores.mean()

In [14]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30
)

[I 2026-09-09 21:26:05,305] A new study created in memory with name: no-name-25f5a872-3993-4b1f-8da5-1e6b4fe0b845
[I 2026-09-09 21:26:10,613] Trial 0 finished with value: 0.6418221051209567 and parameters: {'n_estimators': 249, 'learning_rate': 0.20176020881256912, 'max_depth': 4}. Best is trial 0 with value: 0.6418221051209567.
[I 2026-09-09 21:27:08,406] Trial 1 finished with value: 0.6398786431698563 and parameters: {'n_estimators': 779, 'learning_rate': 0.014927513952834998, 'max_depth': 19}. Best is trial 0 with value: 0.6418221051209567.
[I 2026-09-09 21:27:56,840] Trial 2 finished with value: 0.6513007595288741 and parameters: {'n_estimators': 626, 'learning_rate': 0.04125925905533145, 'max_depth': 18}. Best is trial 2 with value: 0.6513007595288741.
[I 2026-09-09 21:28:12,841] Trial 3 finished with value: 0.6357530150953274 and parameters: {'n_estimators': 195, 'learning_rate': 0.054865558143724576, 'max_depth': 9}. Best is trial 2 with value: 0.6513007595288741.
[I 2026-09-09 

In [15]:
best_params = study.best_params

print("Best parameters:")
print(best_params)

print("\nBest CV accuracy:")
print(study.best_value)

Best parameters:
{'n_estimators': 901, 'learning_rate': 0.11621904727844506, 'max_depth': 10}

Best CV accuracy:
0.6585973071005519


In [16]:
best_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight="balanced",
    random_state=42,
    verbosity=-1,
    **best_params
)

best_model.fit(
    X_train_word2vec,
    y_train_encoded
)

,max_depth,10
,learning_rate,0.11621904727844506
,n_estimators,901
,objective,'multiclass'
,class_weight,'balanced'
,reg_alpha,0.1
,reg_lambda,0.1
,random_state,42
,num_class,3
,verbosity,-1
,boosting_type,'gbdt'


In [17]:
y_pred = best_model.predict(
    X_test_word2vec
)

accuracy = accuracy_score(
    y_test_encoded,
    y_pred
)

print("Test Accuracy:", accuracy)

print(
    classification_report(
        y_test_encoded,
        y_pred
    )
)

Test Accuracy: 0.6575753443338334
              precision    recall  f1-score   support

           0       0.51      0.41      0.45      1650
           1       0.73      0.71      0.72      2529
           2       0.67      0.74      0.70      3154

    accuracy                           0.66      7333
   macro avg       0.63      0.62      0.62      7333
weighted avg       0.65      0.66      0.65      7333

